# 📊 Cathedral — Personal Journey Dashboard

> 🏗️ Build: **2026-05-29 12:56:05**

Every time you ran the **Check** notebook, an event was logged to
`Cathedral_EH.CathedralEvents`. This dashboard reads that history and shows
**how** you got to the cathedral — not just the final score.

Click **▶️ Run all** in the toolbar and read the tiles below.


## Step 1 — Connect to the telemetry table


In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0", "plotly>=5.18.0"],
               check=False, capture_output=True)
import requests, json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

EH_CLUSTER  = "https://trd-z9b3f5xvzm87f8c2kd.z6.kusto.fabric.microsoft.com"
EH_DATABASE = "Cathedral_EH"
EH_TABLE    = "CathedralEvents"

def _kql_token():
    try:
        import notebookutils
        return notebookutils.credentials.getToken("kusto")
    except Exception:
        pass
    try:
        import mssparkutils
        return mssparkutils.credentials.getToken("kusto")
    except Exception:
        pass
    return None

def kql(query: str) -> pd.DataFrame:
    tok = _kql_token()
    if not tok:
        raise RuntimeError("Could not acquire Kusto token.")
    r = requests.post(
        f"{EH_CLUSTER}/v2/rest/query",
        headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
        json={"db": EH_DATABASE, "csl": query},
        timeout=30,
    )
    if r.status_code != 200:
        raise RuntimeError(f"KQL HTTP {r.status_code}: {r.text[:600]}")
    for frame in r.json():
        if frame.get("FrameType") == "DataTable" and frame.get("TableKind") == "PrimaryResult":
            cols = [c["ColumnName"] for c in frame["Columns"]]
            return pd.DataFrame(frame["Rows"], columns=cols)
    return pd.DataFrame()

# Project to friendly snake_case names so the rest of the notebook is readable.
Q = f'''
{EH_TABLE}
| project
    ts          = Timestamp,
    session_id  = SessionId,
    player_id   = PlayerId,
    event_type  = EventType,
    pillar_id   = PillarId,
    pillar_key  = PillarKey,
    pass_fail   = PassFail,
    elegance    = EleganceScore,
    rank        = Rank,
    duration_s  = DurationSeconds
| order by ts asc
'''
df_all = kql(Q)
if df_all.empty:
    print("⚠️ No telemetry yet. Run the Check notebook first.")
else:
    df_all["ts"] = pd.to_datetime(df_all["ts"])
    df_all["elegance"] = pd.to_numeric(df_all["elegance"], errors="coerce")
    df_all["pillar_id"] = pd.to_numeric(df_all["pillar_id"], errors="coerce").astype("Int64")
    print(f"✅ Loaded {len(df_all)} events across {df_all['session_id'].nunique()} session(s).")


## Step 2 — Where you are now (latest attempt per pillar)


In [ ]:
if df_all.empty:
    print("No data.")
else:
    # For each pillar take the LAST event of type 'check' (the user's most recent grading).
    last_check = (
        df_all[df_all["event_type"] == "check"]
        .sort_values("ts")
        .groupby("pillar_id", as_index=False)
        .tail(1)
        .sort_values("pillar_id")
    )

    passed = (last_check["pass_fail"] == "PASS").sum()
    total  = 12
    rank   = last_check.iloc[-1]["rank"] if not last_check.empty else "Stonemason"

    print(f"🏛️  Pillars passed   : {passed} / {total}")
    print(f"📜  Current rank      : {rank}")
    print(f"🎯  Avg elegance      : {last_check[last_check['pass_fail']=='PASS']['elegance'].mean():.1f}")
    print()

    status_color = {"PASS": "#3CB371", "FAIL": "#E5736A", "MISSING": "#BBBBBB"}
    fig = px.bar(
        last_check,
        x="pillar_id", y="elegance",
        color="pass_fail",
        color_discrete_map=status_color,
        hover_data=["pillar_key", "pass_fail", "elegance"],
        title="Elegance per pillar (latest attempt) — taller = leaner DAX",
        labels={"pillar_id": "Pillar #", "elegance": "Elegance (0-100)"},
    )
    fig.update_yaxes(range=[0, 105])
    fig.update_layout(height=380, showlegend=True, legend_title_text="status")
    fig.show()


## Step 3 — Timeline (every attempt, in order)


In [ ]:
if df_all.empty:
    print("No data.")
else:
    df_tl = df_all.copy()
    df_tl["status"] = df_tl["pass_fail"]
    status_color = {"PASS": "#3CB371", "FAIL": "#E5736A", "MISSING": "#BBBBBB"}

    fig = px.scatter(
        df_tl,
        x="ts", y="pillar_id",
        color="status",
        color_discrete_map=status_color,
        symbol="event_type",
        symbol_map={"check": "circle", "calcgroup": "diamond"},
        hover_data=["pillar_key", "elegance", "rank"],
        title="Your attempts over time — circle = measure check, diamond = calc-group check",
        labels={"ts": "Time", "pillar_id": "Pillar #"},
    )
    fig.update_yaxes(dtick=1, range=[0.5, 12.5])
    fig.update_layout(height=420)
    fig.show()

    attempts_per_pillar = (
        df_all[df_all["event_type"] == "check"]
        .groupby("pillar_id").size().rename("attempts").reset_index()
    )
    if (attempts_per_pillar["attempts"] > 1).any():
        grinders = attempts_per_pillar[attempts_per_pillar["attempts"] > 1]
        print("🔁 Pillars you re-tried (= where DAX is hard):")
        for _, row in grinders.iterrows():
            print(f"   #{int(row['pillar_id']):2d}  {int(row['attempts'])} attempts")


## Step 4 — The point of the lesson: Measures vs Calculation Group


In [ ]:
if df_all.empty:
    print("No data.")
else:
    last_check_pass = (
        df_all[(df_all["event_type"] == "check") & (df_all["pass_fail"] == "PASS")]
        .sort_values("ts").groupby("pillar_id", as_index=False).tail(1)
        [["pillar_id", "pillar_key", "elegance"]]
        .rename(columns={"elegance": "Measures"})
    )
    last_cg = (
        df_all[(df_all["event_type"] == "calcgroup") & (df_all["pass_fail"] == "PASS")]
        .sort_values("ts").groupby("pillar_id", as_index=False).tail(1)
        [["pillar_id", "elegance"]]
        .rename(columns={"elegance": "CalcGroup"})
    )

    cmp = last_check_pass.merge(last_cg, on="pillar_id", how="left")
    if cmp["CalcGroup"].isna().all():
        print("ℹ️  The Calc Group challenge hasn't been completed yet.")
        print("    Build the 'Time Intelligence' calc group, run check_calc_group(),")
        print("    then re-run this notebook to see the elegance gain.")
    else:
        cmp_long = cmp.melt(
            id_vars=["pillar_id", "pillar_key"],
            value_vars=["Measures", "CalcGroup"],
            var_name="approach", value_name="elegance",
        ).dropna()

        fig = px.bar(
            cmp_long,
            x="pillar_id", y="elegance",
            color="approach", barmode="group",
            color_discrete_map={"Measures": "#7F8FA6", "CalcGroup": "#F6B93B"},
            title="Elegance gain — same KPI, two implementations",
            labels={"pillar_id": "Pillar #", "elegance": "Elegance"},
        )
        fig.update_yaxes(range=[0, 105])
        fig.update_layout(height=380)
        fig.show()

        gain = (cmp["CalcGroup"] - cmp["Measures"]).dropna()
        if len(gain):
            print(f"📈 Average elegance gain : +{gain.mean():.1f} points")
            print(f"🏆 Biggest single gain   : +{gain.max():.1f} points  "
                  f"(pillar #{int(cmp.loc[gain.idxmax(),'pillar_id'])} — "
                  f"{cmp.loc[gain.idxmax(),'pillar_key']})")


## Step 5 — Session recap


In [ ]:
if df_all.empty:
    print("No data.")
else:
    last_sid = df_all.sort_values("ts").iloc[-1]["session_id"]
    sdf = df_all[df_all["session_id"] == last_sid].sort_values("ts")
    t0, t1 = sdf["ts"].min(), sdf["ts"].max()
    dur = (t1 - t0).total_seconds()

    print(f"🎮 Session ID     : {last_sid}")
    print(f"🧑 Player         : {sdf.iloc[0]['player_id']}")
    print(f"⏱️  Duration       : {dur/60:.1f} min")
    print(f"📊 Events         : {len(sdf)}  "
          f"(check={ (sdf['event_type']=='check').sum() }, "
          f"calcgroup={ (sdf['event_type']=='calcgroup').sum() })")
    print(f"🏛️  Final rank     : {sdf.iloc[-1]['rank']}")
    print()
    print("Per-pillar recap (latest event of this session):")
    recap = sdf.sort_values("ts").groupby("pillar_id").tail(1)[
        ["pillar_id", "pillar_key", "event_type", "pass_fail", "elegance"]
    ].sort_values("pillar_id").reset_index(drop=True)
    print(recap.to_string(index=False))
